In [ ]:
!nvidia-smi

Thu Sep 25 04:15:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = os.getcwd()
print("HOME:", HOME)

HOME: /content


## Install Segment Anything Model (SAM) and other dependencies

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'

  Preparing metadata (setup.py) ... done


In [ ]:
!pip install -q jupyter_bbox_widget roboflow dataclasses-json supervision==0.23.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.5/151.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.4 MB/s eta 0:00:00


### Download SAM weights

In [ ]:
!mkdir -p {HOME}/weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P {HOME}/weights

In [ ]:
import os

CHECKPOINT_PATH = os.path.join(HOME, "weights", "sam_vit_h_4b8939.pth")
print(CHECKPOINT_PATH, "; exist:", os.path.isfile(CHECKPOINT_PATH))

/content/weights/sam_vit_h_4b8939.pth ; exist: True


## Download Example Data

**NONE:** Let's download few example images. Feel free to use your images or videos.

In [ ]:
!mkdir -p {HOME}/data

!wget -q https://media.roboflow.com/notebooks/examples/dog.jpeg -P {HOME}/data
!wget -q https://media.roboflow.com/notebooks/examples/dog-2.jpeg -P {HOME}/data
!wget -q https://media.roboflow.com/notebooks/examples/dog-3.jpeg -P {HOME}/data
!wget -q https://media.roboflow.com/notebooks/examples/dog-4.jpeg -P {HOME}/data

## Load Model

In [ ]:
import torch

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_h"

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)

## Automated Mask Generation

To run automatic mask generation, provide a SAM model to the `SamAutomaticMaskGenerator` class. Set the path below to the SAM checkpoint. Running on CUDA and with the default model is recommended.

In [ ]:
mask_generator = SamAutomaticMaskGenerator(sam)

In [ ]:
import os

IMAGE_NAME = "/content/9P149_SHTURMS_comp_67.jpg"
IMAGE_PATH = os.path.join(HOME, "data", IMAGE_NAME)
IMAGE_PATH = IMAGE_NAME

### Generate masks with SAM

In [ ]:
import cv2
import supervision as sv

image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

sam_result = mask_generator.generate(image_rgb)

### Output format

`SamAutomaticMaskGenerator` returns a `list` of masks, where each mask is a `dict` containing various information about the mask:

* `segmentation` - `[np.ndarray]` - the mask with `(W, H)` shape, and `bool` type
* `area` - `[int]` - the area of the mask in pixels
* `bbox` - `[List[int]]` - the boundary box of the mask in `xywh` format
* `predicted_iou` - `[float]` - the model's own prediction for the quality of the mask
* `point_coords` - `[List[List[float]]]` - the sampled input point that generated this mask
* `stability_score` - `[float]` - an additional measure of mask quality
* `crop_box` - `List[int]` - the crop of the image used to generate this mask in `xywh` format

In [ ]:
print(sam_result[0].keys())

dict_keys(['segmentation', 'area', 'bbox', 'predicted_iou', 'point_coords', 'stability_score', 'crop_box'])


### Results visualisation with Supervision

As of version `0.5.0` Supervision has native support for SAM.

In [ ]:
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)

detections = sv.Detections.from_sam(sam_result=sam_result)

annotated_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

sv.plot_images_grid(
    images=[image_bgr, annotated_image],
    grid_size=(1, 2),
    titles=['source image', 'segmented image']
)

### Interaction with segmentation results

In [ ]:
masks = [
    mask['segmentation']
    for mask
    in sorted(sam_result, key=lambda x: x['area'], reverse=True)
]

sv.plot_images_grid(
    images=masks,
    grid_size=(8, int(len(masks) / 8)),
    size=(16, 16)
)

## Generate Segmentation with Bounding Box

The `SamPredictor` class provides an easy interface to the model for prompting the model. It allows the user to first set an image using the `set_image` method, which calculates the necessary image embeddings. Then, prompts can be provided via the `predict` method to efficiently predict masks from those prompts. The model can take as input both point and box prompts, as well as masks from the previous iteration of prediction.

In [ ]:
mask_predictor = SamPredictor(sam)

In [ ]:
import os

IMAGE_NAME = "dog.jpeg"
IMAGE_PATH = os.path.join(HOME, "data", IMAGE_NAME)
IMAGE_PATH = "/content/tank3_830.jpg"

### Draw Box



In [ ]:
# helper function that loads an image before adding it to the widget

import base64

def encode_image(filepath):
    with open(filepath, 'rb') as f:
        image_bytes = f.read()
    encoded = str(base64.b64encode(image_bytes), 'utf-8')
    return "data:image/jpg;base64,"+encoded

**NOTE:** Execute cell below and use your mouse to draw bounding box on the image 👇

In [ ]:
IS_COLAB = True

if IS_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

from jupyter_bbox_widget import BBoxWidget

widget = BBoxWidget()
widget.image = encode_image(IMAGE_PATH)
widget

BBoxWidget(colors=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#b…

In [ ]:
widget.bboxes

[{'x': 685, 'y': 694, 'width': 670, 'height': 284, 'label': ''}]

### Generate masks with SAM

**NOTE:** `SamPredictor.predict` method takes `np.ndarray` `box` argument in `[x_min, y_min, x_max, y_max]` format. Let's reorganise your data first

In [ ]:
def read_yolo_labels_from_file(file_path, img_width, img_height):
    """
    Reads all YOLO label lines from a file and converts them to
    absolute bounding boxes.

    Args:
        file_path (str): The path to the YOLO .txt label file.
        img_width (int): The width of the image.
        img_height (int): The height of the image.

    Returns:
        list: A list of dictionaries, where each dict contains the
              'class_id' and the 'bbox_array'.
    """
    processed_bboxes = []

    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return processed_bboxes

    print(f"Reading file: {file_path}...")

    # Open the file and read line by line
    with open(file_path, 'r') as f:
        for line_number, line in enumerate(f, 1):
            # The line contains: <class_id> <x_center> <y_center> <width> <height>
            line = line.strip()
            if not line: # Skip empty lines
                continue

            bbox_array, class_id = yolo_line_to_bbox(line, img_width, img_height)

            if bbox_array is not None:
                processed_bboxes.append({
                    'class_id': class_id,
                    'bbox_array': bbox_array
                })
            else:
                print(f"Warning: Skipped line {line_number} due to parsing error: '{line}'")

    return processed_bboxes

In [ ]:
def yolo_line_to_bbox(yolo_line, img_width, img_height):
    """
    Converts a single YOLO label line to an absolute bounding box
    array([x_min, y_min, x_max, y_max]).
    """
    try:
        parts = yolo_line.strip().split()
        class_id = int(parts[0])
        x_c_n, y_c_n, w_n, h_n = map(float, parts[1:]) # normalized coords
    except (ValueError, IndexError):
        # Skip empty or improperly formatted lines
        return None, None

    # Denormalize
    x_c = x_c_n * img_width
    y_c = y_c_n * img_height
    w = w_n * img_width
    h = h_n * img_height

    # Convert (cx, cy, w, h) to (x_min, y_min, x_max, y_max)
    x_min = x_c - (w / 2)
    y_min = y_c - (h / 2)
    x_max = x_c + (w / 2)
    y_max = y_c + (h / 2)

    # Round to integers and format as a NumPy array
    bbox = np.array([
        int(round(x_min)),
        int(round(y_min)),
        int(round(x_max)),
        int(round(y_max))
    ])

    return bbox, class_id


In [ ]:
import cv2

In [ ]:
IMAGE_PATH

'/content/tank3_830.jpg'

In [ ]:
image = cv2.imread(IMAGE_PATH)

In [ ]:
IMAGE_HEIGHT,IMAGE_WIDTH,_ = image.shape

In [ ]:
IMAGE_HEIGHT,IMAGE_WIDTH

(1080, 1920)

In [ ]:
IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080

In [ ]:
all_bboxes = read_yolo_labels_from_file(r'/content/9P149_SHTURMS_comp_67.txt', IMAGE_WIDTH, IMAGE_HEIGHT)
all_bboxes[0]["bbox_array"]

Reading file: /content/9P149_SHTURMS_comp_67.txt...


In [ ]:
all_bboxes[0]["bbox_array"]

array([ 461,  310, 1411,  770])

In [ ]:
import numpy as np

# default_box is going to be used if you will not draw any box on image above
default_box = {'x': 68, 'y': 247, 'width': 555, 'height': 678, 'label': ''}

box = widget.bboxes[0] if widget.bboxes else default_box
box =  np.array([ 685,  268, 1192,  717])

In [ ]:
import cv2
import numpy as np
import supervision as sv

image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

mask_predictor.set_image(image_rgb)

masks, scores, logits = mask_predictor.predict(
    box=box,
    multimask_output=True
)

### Results visualisation with Supervision

In [ ]:
box_annotator = sv.BoxAnnotator(color=sv.Color.RED, color_lookup=sv.ColorLookup.INDEX)
mask_annotator = sv.MaskAnnotator(color=sv.Color.RED, color_lookup=sv.ColorLookup.INDEX)

detections = sv.Detections(
    xyxy=sv.mask_to_xyxy(masks=masks),
    mask=masks
)
detections = detections[detections.area == np.max(detections.area)]

source_image = box_annotator.annotate(scene=image_bgr.copy(), detections=detections)
segmented_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

sv.plot_images_grid(
    images=[source_image, segmented_image],
    grid_size=(1, 2),
    titles=['source image', 'segmented image']
)

### Interaction with segmentation results

In [ ]:
import supervision as v

sv.plot_images_grid(
    images=masks,
    grid_size=(1, 4),
    size=(16, 4)
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
masks.shape

(3, 1080, 1920)

In [ ]:
plt.imshow(masks[-1])

# mine

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_address = r'/content/drive/MyDrive/messbah/data/val'

In [ ]:
images_adderss = os.path.join(base_address,"images")
labels_adderss = os.path.join(base_address,"labels")

In [ ]:
len(os.listdir(images_adderss)),len(os.listdir(labels_adderss))

(1177, 1193)

In [ ]:
def get_files_with_only_class_zero(folder_path):
    """
    Iterates through all .txt files in a folder and collects the filenames
    where every bounding box in that file belongs to class 0.

    Args:
        folder_path (str): The path to the directory containing the YOLO label files.

    Returns:
        list: A list of strings containing the filenames that meet the criteria.
    """
    # This list will store the names of the matching files
    matching_files = []

    if not os.path.exists(folder_path):
        print(f"Error: Folder not found at {folder_path}")
        return matching_files

    print(f"Checking files in: {folder_path}...")

    # List all files in the directory
    for filename in os.listdir(folder_path):

        if filename.endswith('.txt'):
            file_path = os.path.join(folder_path, filename)

            all_class_zero = True
            found_labels = False # To filter out completely empty files

            try:
                # Open and read the file line by line
                with open(file_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue

                        found_labels = True

                        # YOLO format: <class_id> <x_center> ...
                        parts = line.split()

                        # Check the class ID (the first element)
                        if parts and int(parts[0]) != 0:
                            all_class_zero = False
                            # Stop checking this file immediately
                            break

            except Exception as e:
                print(f"Error reading file {filename}: {e}")
                continue

            # If the file had labels AND all classes were 0, add the filename to the list
            if found_labels and all_class_zero:
                matching_files.append(filename.split(".")[0])
            # elif not found_labels:
                # Optional: print a note about empty files
                # print(f"Note: File was empty: {filename}")

    return matching_files

In [ ]:
result_list = get_files_with_only_class_zero(labels_adderss)


Checking files in: /content/drive/MyDrive/messbah/data/val/labels...


In [ ]:
for i in result_list[450:]:
  print(i)
  # address
  _label_address = os.path.join(labels_adderss,i+".txt")
  _image_address = os.path.join(images_adderss,i+".jpg")


  image_bgr = cv2.imread(_image_address)
  print(type(image_bgr))
  if image_bgr is None:
        print(f"Skipping: Failed to load image at {_image_address}")
        continue
  image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
  IMAGE_HEIGHT,IMAGE_WIDTH,_ = image_rgb.shape


  all_bboxes = read_yolo_labels_from_file(_label_address, IMAGE_WIDTH, IMAGE_HEIGHT)
  for j in all_bboxes:
  # all_bboxes[0]["bbox_array"]

    # print(j)
    box = j["bbox_array"]

    masks, scores, logits = mask_predictor.predict(
    box=box,
    multimask_output=True
    )

    uint8_array = (masks[-1] * 255).astype(np.uint8)

    mask_name = i+"_mask.jpg"
    mask_save_address = os.path.join("save",mask_name)
    cv2.imwrite(mask_save_address, uint8_array)


In [ ]:
!zip -r /content/save.zip /content/save

In [ ]:
len(os.listdir("save"))

452

In [ ]:
from google.colab import files
files.download('/content/save.zip')



In [ ]:
files.download('/content/save.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>